# Linear Regression — Real Estate Pricing (California Housing)

We build a robust regression pipeline with proper EDA, feature engineering, cross-validation, and tuning.
Dataset: `sklearn.datasets.fetch_california_housing` (real-world).

**Author:** Olivier Robert-Duboille

**What you'll practice**
- reproducible data loading
- EDA with clear plots
- feature engineering / preprocessing pipelines
- cross-validation + hyperparameter tuning
- baseline vs advanced model comparison

This notebook is part of the *advanced-ml-mastery-collection* and is designed to be reproducible and portfolio-ready.

In [ ]:
# Reproducibility
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Plot settings
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_context('talk')


## Load dataset

Load data into a pandas DataFrame/Series and perform a first sanity check.

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
df.rename(columns={'MedHouseVal': 'target'}, inplace=True)

display(df.head())
print(df.shape)
df.describe().T


## EDA

Explore distributions, relationships, and potential issues (missing values, skew, outliers).

In [ ]:
import pandas as pd

# Missing values
display(df.isna().mean().sort_values(ascending=False).head(10))

# Target distribution
plt.figure(figsize=(8,4))
sns.histplot(df['target'], kde=True, bins=40)
plt.title('Target distribution (Median house value)')
plt.show()

# Correlations
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm', center=0)
plt.title('Feature correlation heatmap')
plt.show()


## Feature engineering

Create domain-inspired features and transformations to improve signal and model fit.

In [ ]:
import numpy as np

df_fe = df.copy()
# A common trick for skewed ratios
df_fe['RoomsPerHousehold'] = df_fe['AveRooms'] / (df_fe['HouseAge'] + 1e-6)
df_fe['BedroomsRatio'] = df_fe['AveBedrms'] / (df_fe['AveRooms'] + 1e-6)
df_fe['PopPerHousehold'] = df_fe['Population'] / (df_fe['AveOccup'] + 1e-6)

# Optional log transform for heavy tails
df_fe['log_Population'] = np.log1p(df_fe['Population'])

display(df_fe.head())


## Train/test split

Create an honest hold-out test set; keep CV strictly on the training data.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_fe.drop(columns=['target'])
y = df_fe['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(X_train.shape, X_test.shape)


## Baselines vs advanced models (CV)

Compare simple baselines to stronger models using cross-validation and appropriate metrics.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_validate

num_cols = X_train.columns.tolist()

preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

ct = ColumnTransformer([('num', preprocess, num_cols)], remainder='drop')

models = {
    'dummy_mean': DummyRegressor(strategy='mean'),
    'linear': LinearRegression(),
    'ridge': Ridge(alpha=1.0, random_state=SEED),
    'rf': RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
    'hgb': HistGradientBoostingRegressor(random_state=SEED),
}

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

scoring = {'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'}

rows = []
for name, model in models.items():
    pipe = Pipeline([('prep', ct), ('model', model)])
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
    rows.append({
        'model': name,
        'rmse_mean': -res['test_rmse'].mean(),
        'rmse_std': res['test_rmse'].std(),
        'r2_mean': res['test_r2'].mean(),
    })

import pandas as pd
cv_df = pd.DataFrame(rows).sort_values('rmse_mean')
display(cv_df)


## Hyperparameter tuning (Ridge + HGB)

Use Grid/Random search with CV to tune key hyperparameters.

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

ridge_pipe = Pipeline([('prep', ct), ('model', Ridge(random_state=SEED))])
ridge_grid = {'model__alpha': np.logspace(-3, 3, 13)}
ridge_gs = GridSearchCV(ridge_pipe, ridge_grid, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_gs.fit(X_train, y_train)
print('Best Ridge RMSE:', -ridge_gs.best_score_)
print('Best Ridge alpha:', ridge_gs.best_params_)

hgb_pipe = Pipeline([('prep', ct), ('model', HistGradientBoostingRegressor(random_state=SEED))])
hgb_space = {
    'model__learning_rate': [0.02, 0.05, 0.1],
    'model__max_depth': [None, 3, 5, 8],
    'model__max_leaf_nodes': [15, 31, 63],
    'model__min_samples_leaf': [20, 50, 100],
}
hgb_rs = RandomizedSearchCV(hgb_pipe, hgb_space, n_iter=25, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1, random_state=SEED)
hgb_rs.fit(X_train, y_train)
print('Best HGB RMSE:', -hgb_rs.best_score_)
print('Best HGB params:', hgb_rs.best_params_)


## Final evaluation + diagnostics

Evaluate on the test set and visualize errors to detect systematic failure modes.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

best_model = hgb_rs.best_estimator_
best_model.fit(X_train, y_train)
pred = best_model.predict(X_test)

rmse = mean_squared_error(y_test, pred, squared=False)
r2 = r2_score(y_test, pred)
print({'rmse': rmse, 'r2': r2})

# Predicted vs actual
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test, y=pred, alpha=0.4)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Predicted vs Actual')
plt.show()

# Residuals
resid = y_test - pred
plt.figure(figsize=(8,4))
sns.histplot(resid, kde=True, bins=40)
plt.title('Residual distribution')
plt.show()
